In [47]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [48]:
np.random.seed(42)
n_rows = np.random.randint(20000, 25000)

# Set the proportion
p_micro = np.random.uniform(0.55, 0.65)
p_small = np.random.uniform(0.10, 0.20)
p_medium = np.random.uniform(0.05, 0.15)
p_large = 1.0 - (p_micro + p_small + p_medium)
prob_scale = [p_micro, p_small, p_medium, p_large]

p_culinary = np.random.uniform(0.15, 0.25)
p_manufacture = np.random.uniform(0.07, 0.13)
p_service = np.random.uniform(0.03, 0.07)
p_trade = np.random.uniform(0.45, 0.55)
p_others = 1.0 - (p_trade + p_culinary + p_manufacture + p_service)
prob_sector = [p_culinary, p_manufacture, p_service, p_trade, p_others]

# Create Dataframe
df = pd.DataFrame()
df['Customer_ID'] = range(100001, 100001 + n_rows)

# Owner Demographic
df['Owner_Age'] = np.random.randint(21, 65, n_rows)
df['Marital_Status'] = np.random.choice(['Single', 'Married', 'Divorced'], p=[0.25, 0.65, 0.10], size=n_rows)
df['Gender'] = np.random.choice(['Male', 'Female'], size=n_rows)

# Business Characteristic
df['Business_Scale'] = np.random.choice(['1. Micro', '2. Small', '3. Medium', '4. Large'], p=prob_scale, size=n_rows)
df['Business_Sector'] = np.random.choice(['1. Culinary', '2. Manufacture', '3. Service', '4. Trade', '5. Others'], p=prob_sector, size=n_rows)
df['Business_Age'] = np.random.randint(1, 25, n_rows)

df['Asset'] = np.zeros(n_rows, dtype=float)
df['Revenue'] = np.zeros(n_rows, dtype=float)
df['Profit'] = np.zeros(n_rows, dtype=float)
df['Num_of_Employee'] = np.zeros(n_rows, dtype=int)
df['Collateral_Value'] = np.zeros(n_rows, dtype=float)
df['Loan_Ceiling'] = np.zeros(n_rows, dtype=float)

mask_Micro = df['Business_Scale'] == '1. Micro'
mask_Small = df['Business_Scale'] == '2. Small'
mask_Medium = df['Business_Scale'] == '3. Medium'
mask_Large = df['Business_Scale'] == '4. Large'

df.loc[mask_Micro, 'Asset'] = np.random.uniform(10, 1000, mask_Micro.sum()) * 1e6
df.loc[mask_Micro, 'Revenue'] = np.random.uniform(50, 2000, mask_Micro.sum()) * 1e6
df.loc[mask_Micro, 'Profit'] = np.random.uniform(0.15, 0.30, mask_Micro.sum()) * df.loc[mask_Micro, 'Revenue']
df.loc[mask_Micro, 'Num_of_Employee'] = np.random.randint(1, 5, mask_Micro.sum())
df.loc[mask_Micro, 'Loan_Ceiling'] = np.random.uniform(5, 100, mask_Micro.sum()) * 1e6

df.loc[mask_Small, 'Asset'] = np.random.uniform(1000, 5000, mask_Small.sum()) * 1e6
df.loc[mask_Small, 'Revenue'] = np.random.uniform(2000, 15000, mask_Small.sum()) * 1e6
df.loc[mask_Small, 'Profit'] = np.random.uniform(0.10, 0.20, mask_Small.sum()) * df.loc[mask_Small, 'Revenue']
df.loc[mask_Small, 'Num_of_Employee'] = np.random.randint(5, 20, mask_Small.sum())
df.loc[mask_Small, 'Loan_Ceiling'] = np.random.uniform(100, 500, mask_Small.sum()) * 1e6

df.loc[mask_Medium, 'Asset'] = np.random.uniform(5000, 10000, mask_Medium.sum()) * 1e6
df.loc[mask_Medium, 'Revenue'] = np.random.uniform(15000, 50000, mask_Medium.sum()) * 1e6
df.loc[mask_Medium, 'Profit'] = np.random.uniform(0.08, 0.15, mask_Medium.sum()) * df.loc[mask_Medium, 'Revenue']
df.loc[mask_Medium, 'Num_of_Employee'] = np.random.randint(20, 100, mask_Medium.sum())
df.loc[mask_Medium, 'Loan_Ceiling'] = np.random.uniform(500, 5000, mask_Medium.sum()) * 1e6

df.loc[mask_Large, 'Asset'] = np.random.uniform(10000, 50000, mask_Large.sum()) * 1e6
df.loc[mask_Large, 'Revenue'] = np.random.uniform(50000, 150000, mask_Large.sum()) * 1e6
df.loc[mask_Large, 'Profit'] = np.random.uniform(0.05, 0.30, mask_Large.sum()) * df.loc[mask_Large, 'Revenue']
df.loc[mask_Large, 'Num_of_Employee'] = np.random.randint(100, 500, mask_Large.sum())
df.loc[mask_Large, 'Loan_Ceiling'] = np.random.uniform(5000, 10000, mask_Large.sum()) * 1e6

mask_Medium_Large = df['Business_Scale'].isin(['3. Medium', '4. Large'])
collateral_multiplier = np.random.uniform(1.2, 2.0, mask_Medium_Large.sum())
df.loc[mask_Medium_Large, 'Collateral_Value'] = df.loc[mask_Medium_Large, 'Loan_Ceiling'] * collateral_multiplier

df['Interest_Rate'] = np.random.uniform(0.06, 0.14, n_rows)

# Calculate Target NPL
df['Prob_Non_Performing'] = 0.01

scale_Micro = df['Business_Scale'] == '1. Micro'
df.loc[scale_Micro, 'Prob_Non_Performing'] += 0.04

sektor_culinary = df['Business_Sector'] == '1. Culinary'
df.loc[sektor_culinary, 'Prob_Non_Performing'] += 0.05

df['Prob_Non_Performing'] += 1/df['Business_Age']/5
df['Prob_Non_Performing'] += df['Loan_Ceiling']/df['Profit'] / 10
df['Prob_Non_Performing'] = np.clip(df['Prob_Non_Performing'], 0, 1)

# Calculate Credit Status (0 = Performing, 1 = Non-performing)
df['Credit_Status'] = np.random.binomial(n=1, p=df['Prob_Non_Performing'])

# Drop probability column
df = df.drop(columns=['Prob_Non_Performing'])

# Export to CSV
df.to_csv('MSME_credit_data.csv', index=False)

In [49]:
np.random.seed(123)
n_rows = np.random.randint(30000, 50000)

p_micro = np.random.uniform(0.50, 0.60)
p_small = np.random.uniform(0.20, 0.25)
p_medium = np.random.uniform(0.10, 0.15)
p_large = 1.0 - (p_micro + p_small + p_medium)
prob_scale = [p_micro, p_small, p_medium, p_large]

p_culinary = np.random.uniform(0.20, 0.30)
p_manufacture = np.random.uniform(0.05, 0.10)
p_service = np.random.uniform(0.05, 0.10)
p_trade = np.random.uniform(0.40, 0.50)
p_others = 1.0 - (p_trade + p_culinary + p_manufacture + p_service)
prob_sector = [p_culinary, p_manufacture, p_service, p_trade, p_others]

df = pd.DataFrame()
df['Customer_ID'] = range(100001, 100001 + n_rows)

df['Owner_Age'] = np.random.randint(21, 65, n_rows)
df['Marital_Status'] = np.random.choice(['Single', 'Married', 'Divorced'], p=[0.20, 0.70, 0.10], size=n_rows)
df['Gender'] = np.random.choice(['Male', 'Female'], size=n_rows)

df['Business_Scale'] = np.random.choice(['1. Micro', '2. Small', '3. Medium', '4. Large'], p=prob_scale, size=n_rows)
df['Business_Sector'] = np.random.choice(['1. Culinary', '2. Manufacture', '3. Service', '4. Trade', '5. Others'], p=prob_sector, size=n_rows)
df['Business_Age'] = np.random.randint(1, 25, n_rows)

df['Asset'] = np.zeros(n_rows, dtype=float)
df['Revenue'] = np.zeros(n_rows, dtype=float)
df['Profit'] = np.zeros(n_rows, dtype=float)
df['Num_of_Employee'] = np.zeros(n_rows, dtype=int)
df['Collateral_Value'] = np.zeros(n_rows, dtype=float)
df['Loan_Ceiling'] = np.zeros(n_rows, dtype=float)

mask_micro = df['Business_Scale'] == '1. Micro'
mask_small = df['Business_Scale'] == '2. Small'
mask_medium = df['Business_Scale'] == '3. Medium'
mask_large = df['Business_Scale'] == '4. Large'

df.loc[mask_micro, 'Asset'] = np.random.uniform(10, 1000, mask_micro.sum()) * 1e6
df.loc[mask_micro, 'Revenue'] = np.random.uniform(50, 2000, mask_micro.sum()) * 1e6
df.loc[mask_micro, 'Profit'] = np.random.uniform(0.15, 0.30, mask_micro.sum()) * df.loc[mask_micro, 'Revenue']
df.loc[mask_micro, 'Num_of_Employee'] = np.random.randint(1, 5, mask_micro.sum())
df.loc[mask_micro, 'Loan_Ceiling'] = np.random.uniform(5, 100, mask_micro.sum()) * 1e6

df.loc[mask_small, 'Asset'] = np.random.uniform(1000, 5000, mask_small.sum()) * 1e6
df.loc[mask_small, 'Revenue'] = np.random.uniform(2000, 15000, mask_small.sum()) * 1e6
df.loc[mask_small, 'Profit'] = np.random.uniform(0.10, 0.20, mask_small.sum()) * df.loc[mask_small, 'Revenue']
df.loc[mask_small, 'Num_of_Employee'] = np.random.randint(5, 20, mask_small.sum())
df.loc[mask_small, 'Loan_Ceiling'] = np.random.uniform(100, 500, mask_small.sum()) * 1e6

df.loc[mask_medium, 'Asset'] = np.random.uniform(5000, 10000, mask_medium.sum()) * 1e6
df.loc[mask_medium, 'Revenue'] = np.random.uniform(15000, 50000, mask_medium.sum()) * 1e6
df.loc[mask_medium, 'Profit'] = np.random.uniform(0.08, 0.15, mask_medium.sum()) * df.loc[mask_medium, 'Revenue']
df.loc[mask_medium, 'Num_of_Employee'] = np.random.randint(20, 100, mask_medium.sum())
df.loc[mask_medium, 'Loan_Ceiling'] = np.random.uniform(500, 5000, mask_medium.sum()) * 1e6

df.loc[mask_large, 'Asset'] = np.random.uniform(10000, 50000, mask_large.sum()) * 1e6
df.loc[mask_large, 'Revenue'] = np.random.uniform(50000, 150000, mask_large.sum()) * 1e6
df.loc[mask_large, 'Profit'] = np.random.uniform(0.05, 0.30, mask_large.sum()) * df.loc[mask_large, 'Revenue']
df.loc[mask_large, 'Num_of_Employee'] = np.random.randint(100, 500, mask_large.sum())
df.loc[mask_large, 'Loan_Ceiling'] = np.random.uniform(5000, 10000, mask_large.sum()) * 1e6

mask_medium_large = df['Business_Scale'].isin(['3. Medium', '4. Large'])
collateral_multiplier = np.random.uniform(1.2, 2.0, mask_medium_large.sum())
df.loc[mask_medium_large, 'Collateral_Value'] = df.loc[mask_medium_large, 'Loan_Ceiling'] * collateral_multiplier

df.drop(columns=['Credit_Status']).to_csv('New_MSME_credit_application.csv', index=False)

In [ ]:
df = pd.read_csv('New_MSME_credit_reviewed.csv')

df = df[df['Approval_Status'] == 'Denied']

df['Prob_Non_Performing'] = np.random.uniform(0.009, 0.011)

scale_micro = df['Business_Scale'] == '1. Micro'
df.loc[scale_micro, 'Prob_Non_Performing'] += np.random.uniform(0.03,0.05)

sektor_culinary = df['Business_Sector'] == '1. Culinary'
df.loc[sektor_culinary, 'Prob_Non_Performing'] += np.random.uniform(0.04, 0.06)

df['Prob_Non_Performing'] += 1/df['Business_Age']/np.random.uniform(9,11)
df['Prob_Non_Performing'] += df['Loan_Ceiling'] / df['Profit'] / np.random.uniform(20,30)
df['Prob_Non_Performing'] = np.clip(df['Prob_Non_Performing'],0,1)

df['Credit_Status'] = np.random.binomial(n=1, p=df['Prob_Non_Performing'])

df[['Customer_ID', 'Credit_Status']].to_csv('New_MSME_credit_data.csv', index=False)